### University of Virginia
### DS 5110: Big Data Systems

### Lab: Supervised Learning
### Last updated: March 2, 2023

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [35]:
#importing SparkSession, VectorAssembler, StandardScaler, LogisticRegression, BinaryClassificationEvaluator
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [32]:
DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

In [33]:
#Reading data
df = spark.read.csv(DATA_FILEPATH, header=True, inferSchema=True)

#### Enter code and solution

In [34]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="diagnosis", outputCol="label")
df = indexer.fit(df).transform(df)

assembler = VectorAssembler(
    inputCols=["f1", "f2"],
    outputCol="features_unscaled"
)

df = assembler.transform(df)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df)
df = scaler_model.transform(df)

#60% training 40% test split
train, test = df.randomSplit([0.6, 0.4], seed=314)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    fitIntercept=True
)

lr_model = lr.fit(train)

predictions = lr_model.transform(test)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)

print("Area under ROC curve for test set:", auc)

Area under ROC curve for test set: 0.9514088556641738


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [22]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [23]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution

In [24]:
from pyspark.sql.functions import col
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

df2 = spark.read.csv(DATA_FILEPATH2, header=True, inferSchema=True)

df2 = df2.withColumn(
    "median_house_value",
    col("median_house_value") / 100000
)


df2 = df2.withColumn(
    "rooms_per_household",
    col("total_rooms") / col("households")
)


#80-20 split
train, test = df2.randomSplit([0.8, 0.2], seed=314)

feats = [
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "rooms_per_household"
]

assembler = VectorAssembler(
    inputCols=feats,
    outputCol="features_unscaled"
)

train = assembler.transform(train)
test = assembler.transform(test)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(train)
train = scaler_model.transform(train)
test = scaler_model.transform(test)

lr = LinearRegression(
    featuresCol="features",
    labelCol="median_house_value",
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8
)

model = lr.fit(train)

predictions = model.transform(test)

evaluator = RegressionEvaluator(
    labelCol="median_house_value",
    predictionCol="prediction",
    metricName="mse"
)

mse = evaluator.evaluate(predictions)

print("Test MSE:", mse)

26/07/03 15:43:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Test MSE: 0.7551749809614869
